# Feature Selection

## Objective

The objective of this notebook is to identify the most relevant features from the original airline dataset for business analytics and machine learning.

The selection is based on the following criteria:

- Business relevance
- Predictive usefulness
- Data availability before departure
- Missing value analysis
- Redundant and duplicate information
- Administrative identifiers
- Derived attributes
- Diversion-specific attributes

Based on these criteria, the original dataset containing 120 columns is reduced to 29 + 6 = 35 business-relevant columns for downstream analytics, Gold layer creation and predictive modelling.

In [1]:
#Import Libraries
from pyspark.sql import SparkSession
from pyspark.sql.functions import *

Starting Spark application


ID,YARN Application ID,Kind,State,Spark UI,Driver log,Current session?
1,application_1784463471983_0002,pyspark,idle,Link,Link,✔


SparkSession available as 'spark'.


## Read Silver Layer

In [2]:
spark = SparkSession.builder \
    .appName("Feature_Selection") \
    .getOrCreate()

df = spark.read.parquet(
    "s3://airline-dataset-2020-2025/Silver/Flight_Data_2020_2025/"
)

## Dataset Shape

In [3]:
print("Rows :",df.count())
print("Columns :",len(df.columns))

('Rows :', 40910253)
('Columns :', 120)

## Feature Selection Criteria

The following rules were used while selecting the final features:

- Retain columns required for business KPIs.
- Retain columns useful for machine learning.
- Retain columns available before or during flight operations.
- Remove duplicate identifiers.
- Remove administrative reference columns.
- Remove derived columns that can be recreated.
- Remove diversion-specific columns containing excessive missing values.
- Remove columns with little or no analytical value.

In [4]:
selected_columns = [

"Year",
"Quarter",
"Month",
"DayofMonth",
"DayOfWeek",
"FlightDate",
"Marketing_Airline_Network",
"Flight_Number_Marketing_Airline",
"Origin",
"OriginState",
"Dest",
"DestState",
"CRSDepTime",
"CRSArrTime",
"DepDelay",
"ArrDelay",
"DepDel15",
"ArrDel15",
"CarrierDelay",
"WeatherDelay",
"NASDelay",
"SecurityDelay",
"LateAircraftDelay",
"Cancelled",
"Diverted",
"Distance",
"AirTime",
"TaxiOut",
"TaxiIn"

]

In [5]:
print("Selected Columns :",len(selected_columns))

('Selected Columns :', 29)

In [6]:
removed_columns = [

c for c in df.columns

if c not in selected_columns

]

print("Removed Columns :",len(removed_columns))

('Removed Columns :', 91)

## Selected Columns

In [7]:
for i,c in enumerate(selected_columns,1):

    print(i,c)

(1, 'Year')
(2, 'Quarter')
(3, 'Month')
(4, 'DayofMonth')
(5, 'DayOfWeek')
(6, 'FlightDate')
(7, 'Marketing_Airline_Network')
(8, 'Flight_Number_Marketing_Airline')
(9, 'Origin')
(10, 'OriginState')
(11, 'Dest')
(12, 'DestState')
(13, 'CRSDepTime')
(14, 'CRSArrTime')
(15, 'DepDelay')
(16, 'ArrDelay')
(17, 'DepDel15')
(18, 'ArrDel15')
(19, 'CarrierDelay')
(20, 'WeatherDelay')
(21, 'NASDelay')
(22, 'SecurityDelay')
(23, 'LateAircraftDelay')
(24, 'Cancelled')
(25, 'Diverted')
(26, 'Distance')
(27, 'AirTime')
(28, 'TaxiOut')
(29, 'TaxiIn')

## Removed Columns

In [8]:
for i,c in enumerate(removed_columns,1):

    print(i,c)

(1, 'Operated_or_Branded_Code_Share_Partners')
(2, 'DOT_ID_Marketing_Airline')
(3, 'IATA_Code_Marketing_Airline')
(4, 'Originally_Scheduled_Code_Share_Airline')
(5, 'DOT_ID_Originally_Scheduled_Code_Share_Airline')
(6, 'IATA_Code_Originally_Scheduled_Code_Share_Airline')
(7, 'Flight_Num_Originally_Scheduled_Code_Share_Airline')
(8, 'Operating_Airline')
(9, 'DOT_ID_Operating_Airline')
(10, 'IATA_Code_Operating_Airline')
(11, 'Tail_Number')
(12, 'Flight_Number_Operating_Airline')
(13, 'OriginAirportID')
(14, 'OriginAirportSeqID')
(15, 'OriginCityMarketID')
(16, 'OriginCityName')
(17, 'OriginStateFips')
(18, 'OriginStateName')
(19, 'OriginWac')
(20, 'DestAirportID')
(21, 'DestAirportSeqID')
(22, 'DestCityMarketID')
(23, 'DestCityName')
(24, 'DestStateFips')
(25, 'DestStateName')
(26, 'DestWac')
(27, 'DepTime')
(28, 'DepDelayMinutes')
(29, 'DepartureDelayGroups')
(30, 'DepTimeBlk')
(31, 'WheelsOff')
(32, 'WheelsOn')
(33, 'ArrTime')
(34, 'ArrDelayMinutes')
(35, 'ArrivalDelayGroups')
(36, 'A

## Missing Value Check for Selected Columns

In [9]:
missing_selected = df.select([

sum(col(c).isNull().cast("int")).alias(c)

for c in selected_columns

])

missing_selected.show(vertical=True,truncate=False)

-RECORD 0-----------------------------------
 Year                            | 0        
 Quarter                         | 0        
 Month                           | 0        
 DayofMonth                      | 0        
 DayOfWeek                       | 0        
 FlightDate                      | 0        
 Marketing_Airline_Network       | 0        
 Flight_Number_Marketing_Airline | 1        
 Origin                          | 0        
 OriginState                     | 0        
 Dest                            | 0        
 DestState                       | 0        
 CRSDepTime                      | 0        
 CRSArrTime                      | 0        
 DepDelay                        | 896518   
 ArrDelay                        | 1014879  
 DepDel15                        | 896518   
 ArrDel15                        | 1014879  
 CarrierDelay                    | 33265986 
 WeatherDelay                    | 33265986 
 NASDelay                        | 33265986 
 SecurityD

## If Any Selected Column is Completely Empty Check

In [10]:
missing_dict = missing_selected.first().asDict()

for c,v in missing_dict.items():

    print(c,v)

('TaxiIn', 926416)
('SecurityDelay', 33265986)
('Flight_Number_Marketing_Airline', 1)
('DepDelay', 896518)
('WeatherDelay', 33265986)
('CRSArrTime', 0)
('DayofMonth', 0)
('DayOfWeek', 0)
('Marketing_Airline_Network', 0)
('TaxiOut', 912686)
('Dest', 0)
('ArrDelay', 1014879)
('AirTime', 1014879)
('CarrierDelay', 33265986)
('CRSDepTime', 0)
('Diverted', 0)
('Distance', 0)
('DepDel15', 896518)
('DestState', 0)
('NASDelay', 33265986)
('FlightDate', 0)
('ArrDel15', 1014879)
('Cancelled', 0)
('Quarter', 0)
('Origin', 0)
('OriginState', 0)
('LateAircraftDelay', 33265986)
('Month', 0)
('Year', 0)

## Missing Value Analysis for Selected Features

Missing value analysis was performed on the 29 selected features to ensure they are suitable for downstream analytics and machine learning.

Observations:

- Most business and operational attributes contain complete data.
- Delay-related columns contain null values primarily for cancelled or diverted flights.
- Delay cause columns (CarrierDelay, WeatherDelay, NASDelay, SecurityDelay and LateAircraftDelay) are populated only when a flight experiences a reportable delay.
- Flight_Number_Marketing_Airline contains only one missing value across the entire dataset, which has negligible impact on analysis.

## Verify Business Columns

In [11]:
df.select(

"Cancelled",

"Diverted",

"DepDel15",

"ArrDel15"

).summary().show()

+-------+-------------------+--------------------+------------------+------------------+
|summary|          Cancelled|            Diverted|          DepDel15|          ArrDel15|
+-------+-------------------+--------------------+------------------+------------------+
|  count|           40910253|            40910253|          40013735|          39895374|
|   mean|0.02241697209743484|0.002390354320223...|0.1890711027101069|0.1916079794113473|
| stddev|0.14803530658288594|0.048832781865639927|0.3915650963273635|0.3935662149114937|
|    min|                0.0|                 0.0|               0.0|               0.0|
|    25%|                0.0|                 0.0|               0.0|               0.0|
|    50%|                0.0|                 0.0|               0.0|               0.0|
|    75%|                0.0|                 0.0|               0.0|               0.0|
|    max|                1.0|                 1.0|               1.0|               1.0|
+-------+------------

## Verify Delay Cause Columns

In [12]:
delay_columns=[

"CarrierDelay",

"WeatherDelay",

"NASDelay",

"SecurityDelay",

"LateAircraftDelay"

]

for c in delay_columns:

    print(c)

    df.select(c).summary().show()

CarrierDelay
+-------+------------------+
|summary|      CarrierDelay|
+-------+------------------+
|  count|           7644267|
|   mean|25.263957028188575|
| stddev| 75.45408450244801|
|    min|               0.0|
|    25%|               0.0|
|    50%|               4.0|
|    75%|              23.0|
|    max|            7232.0|
+-------+------------------+

WeatherDelay
+-------+-----------------+
|summary|     WeatherDelay|
+-------+-----------------+
|  count|          7644267|
|   mean|4.279215521906809|
| stddev|34.04622253656452|
|    min|              0.0|
|    25%|              0.0|
|    50%|              0.0|
|    75%|              0.0|
|    max|           2419.0|
+-------+-----------------+

NASDelay
+-------+------------------+
|summary|          NASDelay|
+-------+------------------+
|  count|           7644267|
|   mean|13.147254144838216|
| stddev|31.962116802658198|
|    min|               0.0|
|    25%|               0.0|
|    50%|               0.0|
|    75%|         

## Delay Cause Feature Validation

Summary statistics were generated for all delay cause attributes.

Observations:

- Delay cause columns are populated only for flights with reportable delays.
- Most values are zero, indicating no delay from that specific cause.
- Carrier Delay and Late Aircraft Delay contribute the highest average delay.
- Security Delay occurs infrequently compared to other delay causes.

## Check Selected Dataset

In [13]:
gold_df=df.select(selected_columns)

gold_df.printSchema()

root
 |-- Year: integer (nullable = true)
 |-- Quarter: integer (nullable = true)
 |-- Month: integer (nullable = true)
 |-- DayofMonth: integer (nullable = true)
 |-- DayOfWeek: integer (nullable = true)
 |-- FlightDate: timestamp (nullable = true)
 |-- Marketing_Airline_Network: string (nullable = true)
 |-- Flight_Number_Marketing_Airline: integer (nullable = true)
 |-- Origin: string (nullable = true)
 |-- OriginState: string (nullable = true)
 |-- Dest: string (nullable = true)
 |-- DestState: string (nullable = true)
 |-- CRSDepTime: integer (nullable = true)
 |-- CRSArrTime: integer (nullable = true)
 |-- DepDelay: double (nullable = true)
 |-- ArrDelay: double (nullable = true)
 |-- DepDel15: double (nullable = true)
 |-- ArrDel15: double (nullable = true)
 |-- CarrierDelay: double (nullable = true)
 |-- WeatherDelay: double (nullable = true)
 |-- NASDelay: double (nullable = true)
 |-- SecurityDelay: double (nullable = true)
 |-- LateAircraftDelay: double (nullable = true)
 |-

## Verify Shape

In [15]:
print("Rows :",gold_df.count())

print("Columns :",len(gold_df.columns))

('Rows :', 40910253)
('Columns :', 29)

In [16]:
gold_df.show(5,truncate=False)

+----+-------+-----+----------+---------+-------------------+-------------------------+-------------------------------+------+-----------+----+---------+----------+----------+--------+--------+--------+--------+------------+------------+--------+-------------+-----------------+---------+--------+--------+-------+-------+------+
|Year|Quarter|Month|DayofMonth|DayOfWeek|FlightDate         |Marketing_Airline_Network|Flight_Number_Marketing_Airline|Origin|OriginState|Dest|DestState|CRSDepTime|CRSArrTime|DepDelay|ArrDelay|DepDel15|ArrDel15|CarrierDelay|WeatherDelay|NASDelay|SecurityDelay|LateAircraftDelay|Cancelled|Diverted|Distance|AirTime|TaxiOut|TaxiIn|
+----+-------+-----+----------+---------+-------------------+-------------------------+-------------------------------+------+-----------+----+---------+----------+----------+--------+--------+--------+--------+------------+------------+--------+-------------+-----------------+---------+--------+--------+-------+-------+------+
|2021|3   

## Feature Selection Summary

The original airline dataset contains **120 attributes** describing flight schedules, airline operations, airport information, delays, cancellations, diversions, and flight performance.

To create a cleaner and more efficient dataset for analytics and machine learning, a systematic feature selection process was carried out based on the following criteria:

* Business relevance for airline performance analysis.
* Importance for delay prediction and operational analytics.
* Availability of information before or during flight operations.
* Missing value analysis and data completeness.
* Removal of redundant and duplicate attributes.
* Exclusion of administrative and reference identifier columns.
* Removal of diversion-specific attributes with excessive missing values.
* Elimination of derived features that can be recreated during feature engineering.

### Selection Outcome

* **Original Features:** **120**
* **Business-Relevant Features Selected:** **29**
* **Additional Representation Features Added:** **6**
* **Final Dataset Features:** **35**
* **Removed Features:** **91**

The six additional columns were included to improve data interpretability and business reporting without affecting the feature selection process:

30. OriginCityName
31. DestCityName
32. OriginStateName
33. DestStateName
34. Operating_Airline (for comparison with the marketing airline)
35. Operated_or_Branded_Code_Share_Partners (used to derive the CodeShareFlag)

The final dataset retains all information required for:

* Airline performance analysis
* Flight delay analysis
* Root cause analysis of delays
* Route and airport performance analysis
* Time-series analysis
* Business KPI reporting and dashboards
* Feature engineering
* Gold Layer creation
* Machine learning and deep learning model development

The remaining **91 features** were removed because they consisted of administrative identifiers, duplicate information, diversion-related attributes with a high percentage of missing values, post-flight operational details not required for prediction, or derived attributes that can be recreated when needed. This reduction improves data quality, simplifies downstream processing, reduces storage and computational overhead, and provides a well-structured dataset for analytics and predictive modelling.


# After selecting the features for better representation of data,
## 6 additional columns are added 

30 . OriginCityName

31 . DestCityName

32 . OriginStateName

33 . DestStateName

34 . Operating_Airline (for comparison between marketing airline)

35 . Operated_or_Branded_Code_Share_Partners (provides a CodeShareFlag)

# Conclusion

The feature selection process successfully reduced the original airline dataset from **120 columns to 29 business-relevant features + 6 for easier representation = 35** while preserving the information required for analytics and predictive modelling.

The selected features capture key aspects of airline operations, including:

- Flight scheduling information
- Airline and route details
- Departure and arrival delays
- Delay causes
- Operational status (Cancelled and Diverted)
- Flight distance and duration
- Airport ground operations

The selected dataset is well suited for downstream processing and provides a clean foundation for:

- Feature Engineering
- Gold Layer creation
- Business Intelligence dashboards
- Delay prediction using Deep Neural Networks
